# 05 — TFT-Inspired Model (Simplified Temporal Attention)
**EduForecast** | LSTM + Multi-head Self-Attention | Notebook 5 of 6

> **Architecture note:** This notebook implements a *Simplified Temporal Attention Model* — the core ideas of the Temporal Fusion Transformer (LSTM encoder → multi-head self-attention → feed-forward) realised in pure PyTorch, without the pytorch-forecasting dependency. The model is trained with **random causal masking** so it genuinely supports early-warning prediction at any week.

In [1]:
import copy
import random
import sys
import time
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DIR, FIGURES_DIR, METRICS_DIR, RESULTS_DIR, RANDOM_SEED

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(RANDOM_SEED)

sns.set_style('whitegrid')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 150

# Device: prefer MPS (Apple Silicon) > CUDA > CPU
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print('Environment ready.')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

Environment ready.
PyTorch : 2.12.0
Device  : mps


## 1. Load Processed Data

In [2]:
print('Loading sequential feature files...')

train_df = pd.read_csv(PROCESSED_DIR / 'train_sequential.csv')
test_df  = pd.read_csv(PROCESSED_DIR / 'test_sequential.csv')

print(f'train_sequential : {train_df.shape}')
print(f'test_sequential  : {test_df.shape}')
print(f'columns          : {list(train_df.columns)}')

REG_KEYS  = ['id_student', 'code_module', 'code_presentation']
FEAT_COLS = [
    'weekly_clicks', 'is_active', 'cumulative_clicks',
    'click_trend', 'weeks_since_active',
    'assessment_score', 'assessment_submitted',
]
TARGET  = 'is_at_risk'
N_WEEKS = 40
N_FEAT  = len(FEAT_COLS)
print(f'\nFeature cols ({N_FEAT}): {FEAT_COLS}')

Loading sequential feature files...


train_sequential : (1042400, 12)
test_sequential  : (261320, 12)
columns          : ['id_student', 'code_module', 'code_presentation', 'week', 'weekly_clicks', 'is_active', 'cumulative_clicks', 'click_trend', 'weeks_since_active', 'assessment_score', 'assessment_submitted', 'is_at_risk']

Feature cols (7): ['weekly_clicks', 'is_active', 'cumulative_clicks', 'click_trend', 'weeks_since_active', 'assessment_score', 'assessment_submitted']


## 2. Prepare Data

In [3]:
print('Reshaping to 3-D tensors (registrations × weeks × features)...')

train_df = train_df.sort_values(REG_KEYS + ['week']).reset_index(drop=True)
test_df  = test_df.sort_values(REG_KEYS + ['week']).reset_index(drop=True)

for split, df in [('train', train_df), ('test', test_df)]:
    counts = df.groupby(REG_KEYS).size()
    assert (counts == N_WEEKS).all()
    print(f'{split}: {len(counts):,} registrations × {N_WEEKS} weeks ✓')

n_train = len(train_df) // N_WEEKS
n_test  = len(test_df)  // N_WEEKS

# Labels: stride N_WEEKS picks week 0 of each registration (all weeks share the same label)
y_train = train_df.iloc[::N_WEEKS][TARGET].values.astype(np.float32)
y_test  = test_df.iloc[::N_WEEKS][TARGET].values.astype(np.float32)

# Feature arrays: (n_reg, N_WEEKS, N_FEAT)
X_train_raw = train_df[FEAT_COLS].values.reshape(n_train, N_WEEKS, N_FEAT).astype(np.float32)
X_test_raw  = test_df[FEAT_COLS].values.reshape(n_test,  N_WEEKS, N_FEAT).astype(np.float32)

# Scale: fit on train 2-D, transform both, reshape back
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_raw.reshape(-1, N_FEAT)).reshape(n_train, N_WEEKS, N_FEAT).astype(np.float32)
X_test_sc  = scaler.transform(X_test_raw.reshape(-1, N_FEAT)).reshape(n_test,  N_WEEKS, N_FEAT).astype(np.float32)

# PyTorch tensors
X_train_t = torch.from_numpy(X_train_sc)
y_train_t = torch.from_numpy(y_train)
X_test_t  = torch.from_numpy(X_test_sc)
y_test_t  = torch.from_numpy(y_test)

BATCH_SIZE = 64
train_loader = DataLoader(
    TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
test_loader = DataLoader(
    TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE, shuffle=False,
)

print(f'\nX_train : {X_train_t.shape}  y_train : {y_train_t.shape}  at-risk: {y_train.mean()*100:.1f}%')
print(f'X_test  : {X_test_t.shape}   y_test  : {y_test_t.shape}   at-risk: {y_test.mean()*100:.1f}%')
print(f'Batches : {len(train_loader)} train / {len(test_loader)} test')

Reshaping to 3-D tensors (registrations × weeks × features)...


train: 26,060 registrations × 40 weeks ✓
test: 6,533 registrations × 40 weeks ✓

X_train : torch.Size([26060, 40, 7])  y_train : torch.Size([26060])  at-risk: 31.2%
X_test  : torch.Size([6533, 40, 7])   y_test  : torch.Size([6533])   at-risk: 31.2%
Batches : 408 train / 103 test


## 3. Define TFT-Inspired Model

In [4]:
class TemporalAttentionModel(nn.Module):
    """Simplified TFT-inspired architecture (Simplified Temporal Attention Model).

    Pipeline:
        1. Input projection  Linear(input_size, hidden_size)
        2. LSTM encoder      captures local temporal dependencies
        3. Self-attention    attends globally across all time steps
        4. Residual + LN     stabilises training
        5. Masked mean-pool  aggregates only valid (non-masked) positions
        6. Feed-forward      binary classification head

    Outputs raw logits (no sigmoid); use BCEWithLogitsLoss during training
    and torch.sigmoid() during inference.
    """

    def __init__(
        self,
        input_size: int = 7,
        hidden_size: int = 64,
        num_layers: int = 2,
        num_heads: int = 4,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        # 1. Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
        )
        # 2. LSTM encoder
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # 3. Multi-head self-attention
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        # 4. Layer normalisation
        self.layer_norm = nn.LayerNorm(hidden_size)
        # 5. Feed-forward classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor | None = None,
        return_attention: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        """x: (B, T, F)  key_padding_mask: (B, T) True=ignore  -> logits: (B,)"""
        # 1. Input projection: (B, T, F) -> (B, T, H)
        x_proj = self.input_proj(x)

        # 2. LSTM encoder: (B, T, H) -> (B, T, H)
        lstm_out, _ = self.lstm(x_proj)

        # 3. Self-attention over LSTM outputs
        attn_out, attn_weights = self.attention(
            query=lstm_out,
            key=lstm_out,
            value=lstm_out,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=True,
        )  # attn_out: (B, T, H)  attn_weights: (B, T, T)

        # 4. Residual + layer norm
        norm_out = self.layer_norm(attn_out + lstm_out)  # (B, T, H)

        # 5. Masked mean-pool: average over valid (non-masked) positions
        if key_padding_mask is not None:
            valid = (~key_padding_mask).float().unsqueeze(-1)         # (B, T, 1)
            pooled = (norm_out * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1)
        else:
            pooled = norm_out.mean(dim=1)                             # (B, H)

        # 6. Classifier -> logits: (B,)
        logits = self.classifier(pooled).squeeze(-1)

        if return_attention:
            return logits, attn_weights
        return logits


# Print summary
_probe = TemporalAttentionModel(input_size=N_FEAT)
total_p     = sum(p.numel() for p in _probe.parameters())
trainable_p = sum(p.numel() for p in _probe.parameters() if p.requires_grad)

print('TemporalAttentionModel (TFT-Inspired):')
print(_probe)
print(f'\nTotal parameters     : {total_p:,}')
print(f'Trainable parameters : {trainable_p:,}')
print(f'(LSTM notebook had   : 54,081 parameters)')

TemporalAttentionModel (TFT-Inspired):
TemporalAttentionModel(
  (input_proj): Sequential(
    (0): Linear(in_features=7, out_features=64, bias=True)
    (1): ReLU()
  )
  (lstm): LSTM(64, 64, num_layers=2, batch_first=True, dropout=0.1)
  (attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total parameters     : 85,953
Trainable parameters : 85,953
(LSTM notebook had   : 54,081 parameters)


## 4. Training with Causal Masking

In [5]:
# ── Causal masking: trains the model to predict from any prefix ──────────
MASK_PROB = 0.75  # fraction of training samples that receive random truncation

def apply_causal_mask(
    X_batch: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Randomly truncate each sample in the batch.

    Returns:
        X_masked  : input with future weeks zeroed out  (B, T, F)
        pmask     : key_padding_mask — True = ignore   (B, T)
    """
    B, T, _ = X_batch.shape
    X_masked = X_batch.clone()
    pmask    = torch.zeros(B, T, dtype=torch.bool)
    for i in range(B):
        if random.random() < MASK_PROB:
            t = random.randint(1, T)   # keep weeks 0 .. t-1
            X_masked[i, t:, :] = 0.0
            pmask[i, t:] = True
    return X_masked, pmask

# ── Hyperparameters ───────────────────────────────────────────────────────
EPOCHS   = 30
LR       = 0.001
PATIENCE = 5
BEST_PATH = RESULTS_DIR / 'tft_best_model.pth'

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32)
print(f'pos_weight = {pos_weight.item():.4f}  ({n_neg:,} neg / {n_pos:,} pos)')

model     = TemporalAttentionModel(input_size=N_FEAT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

train_losses, val_losses, val_aucs = [], [], []
best_val_auc = 0.0
best_epoch   = 0
no_improve   = 0

print(f'\nStarting training ({EPOCHS} epochs, early stopping patience={PATIENCE})')
print(f'Causal masking: {MASK_PROB*100:.0f}% of samples randomly truncated each batch')
print(f'{"Epoch":<8} {"TrainLoss":>10} {"ValLoss":>10} {"ValAUC":>9} {"Time":>7}')
print('-' * 50)

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        X_m, pmask = apply_causal_mask(X_b.cpu())
        X_m   = X_m.to(device)
        pmask = pmask.to(device)

        optimizer.zero_grad()
        logits = model(X_m, key_padding_mask=pmask)
        loss   = criterion(logits, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * len(y_b)
    train_loss = running_loss / n_train

    # ── Validate (no masking — full sequences) ───────────────────────────
    model.eval()
    val_running = 0.0
    all_probs   = []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            logits = model(X_b)
            val_running += criterion(logits, y_b).item() * len(y_b)
            all_probs.append(torch.sigmoid(logits).cpu())
    val_loss = val_running / n_test
    val_probs = torch.cat(all_probs).numpy()
    val_auc   = roc_auc_score(y_test, val_probs)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_aucs.append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch   = epoch
        no_improve   = 0
        torch.save(copy.deepcopy(model.state_dict()), BEST_PATH)
    else:
        no_improve += 1

    if epoch % 5 == 0 or epoch == 1:
        elapsed = time.time() - t0
        marker  = ' *' if epoch == best_epoch else ''
        print(f'Epoch {epoch:3d}/{EPOCHS} | {train_loss:10.4f} | {val_loss:10.4f} | '
              f'{val_auc:9.4f} | {elapsed:6.0f}s{marker}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs).')
        break

print(f'\nTraining complete in {time.time()-t0:.1f}s')
print(f'Best epoch: {best_epoch}  |  Best Val AUC: {best_val_auc:.4f}')

pos_weight = 2.2094  (17,940 neg / 8,120 pos)



Starting training (30 epochs, early stopping patience=5)
Causal masking: 75% of samples randomly truncated each batch
Epoch     TrainLoss    ValLoss    ValAUC    Time
--------------------------------------------------


Epoch   1/30 |     0.6124 |     0.4346 |    0.9343 |     10s *


Epoch   5/30 |     0.5362 |     0.4018 |    0.9413 |     35s *


Epoch  10/30 |     0.5244 |     0.3942 |    0.9424 |     66s



Early stopping at epoch 13 (no improvement for 5 epochs).

Training complete in 84.1s
Best epoch: 8  |  Best Val AUC: 0.9433


In [6]:
# ── Plot 1: Training history ───────────────────────────────────────────────
epoch_range = range(1, len(train_losses) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epoch_range, train_losses, color='steelblue', linewidth=2, label='Train Loss')
ax1.plot(epoch_range, val_losses,   color='tomato',    linewidth=2, label='Val Loss')
ax1.axvline(best_epoch, color='green', linestyle='--', linewidth=1.5,
            label=f'Best epoch ({best_epoch})')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('BCEWithLogitsLoss')
ax1.set_title('Train vs Validation Loss'); ax1.legend()

ax2.plot(epoch_range, val_aucs, color='seagreen', linewidth=2, label='Val AUC')
ax2.axvline(best_epoch, color='red', linestyle='--', linewidth=1.5,
            label=f'Best epoch ({best_epoch})')
ax2.axhline(best_val_auc, color='gray', linestyle=':', linewidth=1.2,
            label=f'Best AUC = {best_val_auc:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('AUC-ROC')
ax2.set_title('Validation AUC by Epoch'); ax2.legend()

fig.suptitle('TFT-Inspired Model Training History', fontsize=14)
plt.tight_layout()
out1 = FIGURES_DIR / '05_tft_training.png'
fig.savefig(out1, dpi=DPI, bbox_inches='tight')
plt.close(fig)
print(f'Saved \u2192 {out1}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/05_tft_training.png


## 5. Evaluation

In [7]:
print('Loading best checkpoint...')
model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()

all_probs_list = []
with torch.no_grad():
    for X_b, _ in test_loader:
        logits = model(X_b.to(device))
        all_probs_list.append(torch.sigmoid(logits).cpu())

tft_prob = torch.cat(all_probs_list).numpy()
tft_pred = (tft_prob >= 0.5).astype(int)

tft_acc  = accuracy_score(y_test, tft_pred)
tft_prec = precision_score(y_test, tft_pred, average='weighted', zero_division=0)
tft_rec  = recall_score(y_test, tft_pred,    average='weighted', zero_division=0)
tft_f1   = f1_score(y_test, tft_pred,        average='weighted', zero_division=0)
tft_auc  = roc_auc_score(y_test, tft_prob)
tft_sens = recall_score(y_test, tft_pred, pos_label=1, average='binary', zero_division=0)
tft_spec = recall_score(y_test, tft_pred, pos_label=0, average='binary', zero_division=0)

print()
print('TFT-INSPIRED MODEL RESULTS')
print(f'  Accuracy    : {tft_acc:.4f}')
print(f'  Precision   : {tft_prec:.4f}')
print(f'  Recall      : {tft_rec:.4f}')
print(f'  F1 Score    : {tft_f1:.4f}')
print(f'  AUC-ROC     : {tft_auc:.4f}')
print(f'  Sensitivity : {tft_sens:.4f}  (recall for At-Risk class)')
print(f'  Specificity : {tft_spec:.4f}  (recall for Not-At-Risk class)')
print()
print('Classification Report:')
print(classification_report(y_test, tft_pred, target_names=['Not At-Risk', 'At-Risk']))

Loading best checkpoint...



TFT-INSPIRED MODEL RESULTS
  Accuracy    : 0.8573
  Precision   : 0.8883
  Recall      : 0.8573
  F1 Score    : 0.8619
  AUC-ROC     : 0.9433
  Sensitivity : 0.9519  (recall for At-Risk class)
  Specificity : 0.8145  (recall for Not-At-Risk class)

Classification Report:
              precision    recall  f1-score   support

 Not At-Risk       0.97      0.81      0.89      4497
     At-Risk       0.70      0.95      0.81      2036

    accuracy                           0.86      6533
   macro avg       0.84      0.88      0.85      6533
weighted avg       0.89      0.86      0.86      6533



In [8]:
tft_metrics = {
    'model':       'TFT-Inspired',
    'accuracy':    round(tft_acc,  4),
    'precision':   round(tft_prec, 4),
    'recall':      round(tft_rec,  4),
    'f1':          round(tft_f1,   4),
    'auc_roc':     round(tft_auc,  4),
    'sensitivity': round(tft_sens, 4),
    'specificity': round(tft_spec, 4),
}
pd.DataFrame([tft_metrics]).to_csv(METRICS_DIR / 'tft_metrics.csv', index=False)
print('Saved \u2192 tft_metrics.csv')

Saved → tft_metrics.csv


## 6. Attention Weights Analysis

In [9]:
print('Collecting attention weights on full test set (no masking)...')

model.eval()
attn_list = []
with torch.no_grad():
    for start in range(0, n_test, 512):
        X_b = X_test_t[start:start + 512].to(device)
        _, attn_w = model(X_b, return_attention=True)   # attn_w: (B, T, T)
        attn_list.append(attn_w.cpu().numpy())

all_attn = np.concatenate(attn_list, axis=0)   # (n_test, 40, 40)
print(f'Attention tensor collected: {all_attn.shape}  (registrations × query_weeks × key_weeks)')

# Mean attention received by each KEY week (averaged over all query positions)
# avg_attn[i,j] = mean over query positions of attn[i] -- then average over samples
avg_attn_matrix = all_attn.mean(axis=0)        # (40, 40)  query × key
attn_received   = avg_attn_matrix.mean(axis=0)  # (40,)    importance per key-week

# Split by outcome
at_risk_idx  = np.where(y_test == 1)[0]
no_risk_idx  = np.where(y_test == 0)[0]

attn_risk    = all_attn[at_risk_idx].mean(axis=0).mean(axis=0)   # (40,)
attn_no_risk = all_attn[no_risk_idx].mean(axis=0).mean(axis=0)   # (40,)

# Top 5 attended weeks per group
top5_risk    = np.argsort(attn_risk)[-5:][::-1]
top5_no_risk = np.argsort(attn_no_risk)[-5:][::-1]

print(f'\nTop 5 most attended weeks — At-Risk students:')
for rank, w in enumerate(top5_risk, 1):
    print(f'  {rank}. Week {w:2d}  (attention weight: {attn_risk[w]:.5f})')

print(f'\nTop 5 most attended weeks — Not At-Risk students:')
for rank, w in enumerate(top5_no_risk, 1):
    print(f'  {rank}. Week {w:2d}  (attention weight: {attn_no_risk[w]:.5f})')

peak_risk    = int(np.argmax(attn_risk))
peak_no_risk = int(np.argmax(attn_no_risk))
print(f'\nPeak attention week — At-Risk    : Week {peak_risk}')
print(f'Peak attention week — Not At-Risk : Week {peak_no_risk}')
print(f'Difference in peak week           : {abs(peak_risk - peak_no_risk)} weeks')

Attention tensor collected: (6533, 40, 40)  (registrations × query_weeks × key_weeks)

Top 5 most attended weeks — At-Risk students:
  1. Week 39  (attention weight: 0.02611)
  2. Week 38  (attention weight: 0.02606)
  3. Week 37  (attention weight: 0.02601)
  4. Week 36  (attention weight: 0.02597)
  5. Week 35  (attention weight: 0.02594)

Top 5 most attended weeks — Not At-Risk students:
  1. Week 39  (attention weight: 0.02871)
  2. Week 38  (attention weight: 0.02818)
  3. Week 37  (attention weight: 0.02752)
  4. Week 18  (attention weight: 0.02726)
  5. Week 17  (attention weight: 0.02726)

Peak attention week — At-Risk    : Week 39
Peak attention week — Not At-Risk : Week 39
Difference in peak week           : 0 weeks


In [10]:
# ── Plot 2: Attention heatmap (40×40) ─────────────────────────────────────
# Downsample labels to every 5 weeks for readability
tick_pos   = list(range(0, 40, 5))
tick_labels = [str(w) for w in tick_pos]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(avg_attn_matrix, cmap='viridis', aspect='auto', origin='upper')
plt.colorbar(im, ax=ax, label='Mean Attention Weight')
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel('Key Week (source of information)', fontsize=11)
ax.set_ylabel('Query Week (week making prediction)', fontsize=11)
ax.set_title('TFT Attention Weights — Average Across Test Students', fontsize=13, pad=12)
plt.tight_layout()
out2 = FIGURES_DIR / '05_attention_heatmap.png'
fig.savefig(out2, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out2}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/05_attention_heatmap.png


In [11]:
# ── Plot 3: Attention by outcome — line plot ──────────────────────────────
week_axis = np.arange(N_WEEKS)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(week_axis, attn_risk,    color='#e74c3c', linewidth=2.2,
        label='At-Risk (Withdrawn)')
ax.plot(week_axis, attn_no_risk, color='#2ecc71', linewidth=2.2,
        label='Not At-Risk')

# Annotate peak attention weeks
ax.axvline(peak_risk,    color='#e74c3c', linestyle=':', linewidth=1.2, alpha=0.7)
ax.axvline(peak_no_risk, color='#2ecc71', linestyle=':', linewidth=1.2, alpha=0.7)
ax.text(peak_risk + 0.4,    attn_risk.max()    * 0.98, f'Week {peak_risk}',
        color='#e74c3c', fontsize=9, va='top')
ax.text(peak_no_risk + 0.4, attn_no_risk.max() * 0.98, f'Week {peak_no_risk}',
        color='#27ae60', fontsize=9, va='top')

ax.set_xlabel('Week Number', fontsize=12)
ax.set_ylabel('Mean Attention Weight Received', fontsize=12)
ax.set_title('Attention Weight by Week — At-Risk vs Not At-Risk Students', fontsize=13, pad=12)
ax.legend(fontsize=11)
ax.set_xlim(0, 39)
plt.tight_layout()
out3 = FIGURES_DIR / '05_attention_by_outcome.png'
fig.savefig(out3, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out3}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/05_attention_by_outcome.png


## 7. Temporal AUC Analysis

In [12]:
print('Computing temporal AUC at weeks 1\u201339...')
print('(causal-mask trained model: key_padding_mask tells attention to ignore future weeks)')

RF_AUC = 0.9472
LR_AUC = 0.9375

model.eval()
temporal_aucs: list[tuple[int, float]] = []

t_start = time.time()
for w in range(1, 40):
    # Mask future weeks in scaled-feature space
    X_masked = X_test_t.clone()
    X_masked[:, w:, :] = 0.0
    # Explicit key_padding_mask: weeks w..39 are padding
    pmask = torch.zeros(n_test, N_WEEKS, dtype=torch.bool)
    pmask[:, w:] = True

    probs_w: list[float] = []
    with torch.no_grad():
        for start in range(0, n_test, 512):
            end   = min(start + 512, n_test)
            X_b   = X_masked[start:end].to(device)
            pm_b  = pmask[start:end].to(device)
            p     = torch.sigmoid(model(X_b, key_padding_mask=pm_b)).cpu().numpy()
            probs_w.extend(p.tolist())

    auc_w = roc_auc_score(y_test, probs_w)
    temporal_aucs.append((w, auc_w))

print(f'Done in {time.time()-t_start:.1f}s')

print(f'\n{"Week":>6}  {"TFT AUC":>9}  {"vs RF":>9}  {"vs LSTM":>9}')
print('-' * 40)
# LSTM temporal AUC reference values from notebook 04
lstm_temporal_ref = {5:0.4294, 10:0.2870, 15:0.4092, 20:0.3557,
                     25:0.3120, 30:0.4151, 39:0.9021}
for w, auc_w in temporal_aucs:
    if w in [5, 10, 15, 20, 25, 30, 39]:
        vs_rf   = auc_w - RF_AUC
        vs_lstm = auc_w - lstm_temporal_ref.get(w, float('nan'))
        print(f'Week {w:2d}   {auc_w:.4f}   {vs_rf:+.4f}   {vs_lstm:+.4f}')

exceeds_rf = [(w, a) for w, a in temporal_aucs if a >= RF_AUC]
if exceeds_rf:
    w_cross, a_cross = exceeds_rf[0]
    print(f'\nTFT first exceeds RF baseline at week {w_cross}  (AUC = {a_cross:.4f})')
else:
    w_cross, a_cross = None, None
    print('\nTFT does not exceed RF at any single week.')

Computing temporal AUC at weeks 1–39...
(causal-mask trained model: key_padding_mask tells attention to ignore future weeks)


Done in 8.4s

  Week    TFT AUC      vs RF    vs LSTM
----------------------------------------
Week  5   0.7590   -0.1882   +0.3296
Week 10   0.8382   -0.1090   +0.5512
Week 15   0.8770   -0.0702   +0.4678
Week 20   0.9012   -0.0460   +0.5455
Week 25   0.9128   -0.0344   +0.6008
Week 30   0.9166   -0.0306   +0.5015
Week 39   0.9157   -0.0315   +0.0136

TFT does not exceed RF at any single week.


In [13]:
# ── Plot 4: Temporal AUC — all models ─────────────────────────────────────
weeks_l, aucs_l = zip(*temporal_aucs)

fig, ax = plt.subplots(figsize=(13, 6))

# Baselines: horizontal constant lines (static models use all data at once)
ax.axhline(RF_AUC, color='#e74c3c',  linestyle='--', linewidth=2.0,
           label=f'Random Forest (AUC={RF_AUC}, all features)')
ax.axhline(LR_AUC, color='#e67e22',  linestyle='--', linewidth=2.0,
           label=f'Logistic Regression (AUC={LR_AUC}, all features)')

# LSTM temporal (reference values from notebook 04)
lstm_weeks  = sorted(lstm_temporal_ref.keys())
lstm_aucs_r = [lstm_temporal_ref[w] for w in lstm_weeks]
ax.plot(lstm_weeks, lstm_aucs_r, 'o--', color='#9b59b6', linewidth=1.8,
        markersize=5, label='LSTM (full-seq trained, zero-mask eval)', alpha=0.8)

# TFT temporal curve
ax.plot(weeks_l, aucs_l, 's-', color='#3498db', linewidth=2.3,
        markersize=4, label='TFT-Inspired (causal-mask trained)')

if w_cross is not None:
    ax.axvline(w_cross, color='#3498db', linestyle=':', linewidth=1.4, alpha=0.6)
    ax.annotate(f'Week {w_cross}:\nTFT \u2265 RF',
                xy=(w_cross, a_cross),
                xytext=(w_cross + 1.5, a_cross - 0.03),
                fontsize=9, color='#2980b9',
                arrowprops=dict(arrowstyle='->', color='#2980b9', lw=1.1))

ax.set_xlabel('Week Number', fontsize=12)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('Early Warning Capability — Temporal AUC by Week', fontsize=13, pad=12)
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim(1, 39)
y_min = min(min(aucs_l), min(lstm_aucs_r), LR_AUC) - 0.05
ax.set_ylim(max(0.2, y_min), 1.01)
plt.tight_layout()
out4 = FIGURES_DIR / '05_temporal_auc_all.png'
fig.savefig(out4, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out4}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/05_temporal_auc_all.png


## 8. Final Model Comparison

In [14]:
# Load v2 comparison (LR + RF + LSTM) and append TFT
comp_v2 = pd.read_csv(METRICS_DIR / 'model_comparison_v2.csv')

tft_row = pd.DataFrame([{
    'model':     'TFT-Inspired',
    'accuracy':  round(tft_acc,  4),
    'precision': round(tft_prec, 4),
    'recall':    round(tft_rec,  4),
    'f1':        round(tft_f1,   4),
    'auc_roc':   round(tft_auc,  4),
}])

final_comp = pd.concat([comp_v2, tft_row], ignore_index=True)
final_comp.to_csv(METRICS_DIR / 'final_model_comparison.csv', index=False)
print('Saved \u2192 final_model_comparison.csv')

# Sensitivity values (At-Risk recall)
sensitivity_map = {
    'Logistic Regression': 0.8900,  # from notebook 03 classification report
    'Random Forest':       0.9489,
    'LSTM':                0.9578,
    'TFT-Inspired':        round(tft_sens, 4),
}

rf_auc  = float(comp_v2[comp_v2['model']=='Random Forest']['auc_roc'].iloc[0])
best_model_row = final_comp.loc[final_comp['auc_roc'].idxmax()]

print()
print('=' * 61)
print('FINAL MODEL COMPARISON')
print('=' * 61)
print(f'{"Model":<22} {"AUC":>7} {"F1":>7} {"Acc":>7} {"Sensitivity":>12}')
print('-' * 61)
for _, row in final_comp.iterrows():
    sens = sensitivity_map.get(row['model'], float('nan'))
    sens_str = f'{sens:.4f}' if not np.isnan(sens) else '   —   '
    print(f'{row["model"]:<22} {row["auc_roc"]:>7.4f} {row["f1"]:>7.4f} '
          f'{row["accuracy"]:>7.4f} {sens_str:>12}')
print('=' * 61)
print(f'Best model: {best_model_row["model"]}  (AUC = {best_model_row["auc_roc"]:.4f})')
print(f'AUC improvement over RF: {best_model_row["auc_roc"] - rf_auc:+.4f}')
print('=' * 61)

Saved → final_model_comparison.csv

FINAL MODEL COMPARISON
Model                      AUC      F1     Acc  Sensitivity
-------------------------------------------------------------
Logistic Regression     0.9375  0.8596  0.8560       0.8900
Random Forest           0.9472  0.8629  0.8584       0.9489
LSTM                    0.9436  0.8611  0.8564       0.9578
TFT-Inspired            0.9433  0.8619  0.8573       0.9519
Best model: Random Forest  (AUC = 0.9472)
AUC improvement over RF: +0.0000


In [15]:
# ── Plot 5: Final comparison bar chart — all 4 models ────────────────────
metric_names    = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']
display_metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC-ROC']
palette = ['#3498db', '#e74c3c', '#9b59b6', '#2ecc71']

x      = np.arange(len(metric_names))
n_mdl  = len(final_comp)
width  = 0.19
offsets = np.linspace(-(n_mdl - 1) * width / 2,
                       (n_mdl - 1) * width / 2, n_mdl)

fig, ax = plt.subplots(figsize=(14, 6))
for i, (_, row) in enumerate(final_comp.iterrows()):
    vals = [row[m] for m in metric_names]
    bars = ax.bar(x + offsets[i], vals, width,
                  label=row['model'], color=palette[i], alpha=0.87, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.004,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(display_metrics, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.14)
ax.set_title('EduForecast — Final Model Performance Comparison', fontsize=14, pad=12)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
plt.tight_layout()
out5 = FIGURES_DIR / '05_final_comparison.png'
fig.savefig(out5, dpi=DPI)
plt.close(fig)
print(f'Saved \u2192 {out5}')

print('\nAll figures from this notebook:')
for f in sorted(FIGURES_DIR.glob('05_*.png')):
    print(f'  {f.name}')

Saved → /Users/sergazieff.n/VSCodeProjects/eduforecast/results/figures/05_final_comparison.png

All figures from this notebook:
  05_attention_by_outcome.png
  05_attention_heatmap.png
  05_final_comparison.png
  05_temporal_auc_all.png
  05_tft_training.png
